# 07 · Sensitivity analysis and paired tests (after the test-period run)

Additional analyses requested during manuscript preparation. They **only read saved forecasts and bounds**; no
forecast, bound or result of notebooks 01-06 is changed.

1. **Sensitivity of the proposed bound** to its tuning constants, varied one at a time
   (`spread_sensitivity.py`):
   * spread quantile $q \in \{0.75, 0.80, 0.85, 0.90, 0.95\}$ (paper: 0.90)
   * history window $M/2$, $M$, $2M$ (paper: 60 / 104 / 252)
   * minimum history $m/2$, $m$, $2m$ (paper: 12 / 26 / 60)
2. **Paired tests** of the proposed bound against pooled split conformal on the same instruments
   (`paired_tests.py`, optional): exact McNemar test, paired difference with CI, Wilcoxon signed-rank test.

Keep `config/config.yaml` exactly as in the test-period run (`mode: holdout`). CPU runtime is enough.
Approximate time: 45-90 minutes for the sensitivity analysis (mostly the daily horizon), 5-10 minutes for the paired
tests. If Colab disconnects, re-run from the top.

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

## Locate the scripts
The scripts must run from the `scripts/` folder (they find the project folder from their own location). If you placed them in `src/calvol/`, they are copied to `scripts/` here.

In [ ]:
import shutil
from pathlib import Path
SCRIPTS = Path(PROJECT_DIR) / "scripts"
SCRIPTS.mkdir(exist_ok=True)
found = {}
for name in ["spread_sensitivity.py", "paired_tests.py"]:
    for loc in [SCRIPTS / name, Path(PROJECT_DIR) / "src" / "calvol" / name, Path(PROJECT_DIR) / name]:
        if loc.exists():
            if loc.parent != SCRIPTS:
                shutil.copy(loc, SCRIPTS / name)
                print(f"copied {loc.relative_to(PROJECT_DIR)} -> scripts/{name}")
            found[name] = SCRIPTS / name
            break
    print(("FOUND   " if name in found else "MISSING ") + name)
assert "spread_sensitivity.py" in found, "Place spread_sensitivity.py in scripts/ or src/calvol/"
assert MODE == "holdout", "Run this notebook with the same config as the test-period run (mode: holdout)"
for f in ["predictions/assets_h21.parquet", "bounds/assets_h21.parquet"]:
    print(("OK      " if (P.results / f).exists() else "MISSING ") + str(P.results / f))

## 1 · Sensitivity analysis
Output is streamed below; each line reports one finished setting.

In [ ]:
def run(script, *args):
    p = subprocess.Popen([sys.executable, str(script), *args], cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    assert p.returncode == 0, f"{script.name} failed (see output above)"

run(found["spread_sensitivity.py"])

### Reproduction check
For the paper's settings the recomputed bounds must equal the saved bounds (differences below about 1e-5 come from float32 storage).

In [ ]:
chk = pd.read_csv(P.tables / "sensitivity_check.csv")
display(chk)
print("max difference:", chk["max_abs_diff_vs_saved"].max())
assert chk["max_abs_diff_vs_saved"].max() < 1e-4, "recomputed bounds do not match the saved bounds"

### Summary: share of asset-level Kupiec tests rejected (%), summed over the four levels

In [ ]:
s = pd.read_csv(P.tables / "sensitivity_summary.csv")
for per in ["development", "test"]:
    d = s[s["period"] == per]
    display(Markdown(f"**{per} period**"))
    display(d.pivot_table(index=["varied", "setting"], columns="h", values="reject_share_pct").round(1))
display(Markdown("**Mean width of the bound (test period)**"))
display(s[s["period"] == "test"].pivot_table(index=["varied", "setting"], columns="h", values="width").round(3))

## 2 · Paired tests (optional)

In [ ]:
if "paired_tests.py" in found:
    run(found["paired_tests.py"])
    display(pd.read_csv(P.tables / "paired_tests_pooled_split.csv").round(4))
else:
    print("paired_tests.py not found - skipped")

## Files to send
The archive below contains the new tables only.

In [ ]:
import zipfile
z = P.results_root / "calvol_sensitivity_results.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(P.tables.glob("sensitivity*")) + sorted(P.tables.glob("paired_tests*")):
        zf.write(f, f.relative_to(P.results_root))
        print("added", f.name)
print("Archive:", z)
if IN_COLAB:
    from google.colab import files
    files.download(str(z))